In [1]:
import numpy as np

In [2]:
# functions

def sigmoid(z): 
	return 1 / (1 + np.exp(-z))

the class the manages individual layers

In [3]:
class DenseLayer:
    # the input dim is the (n) and the output is (h). for notations look into the forward pass doc.
    def __init__(self, input_dim, output_dim, activation_function):
        self.W = np.random.randn(input_dim, output_dim) * 0.01
        self.b = np.zeros((1, output_dim))
        self.activation_function = activation_function.lower()

        self.dZ = None
        self.dW = None
        self.db = None

    def forward(self, A_prev):
        self.A_prev = A_prev
        self.Z = np.dot(self.A_prev, self.W) + self.b

        if self.activation_function == "relu":
            self.A = np.maximum(0, self.Z)
        elif self.activation_function == "sigmoid":
            self.A = sigmoid(self.Z)

        return self.A


    def backward(self, dA):

        if self.activation_function == "relu":
            self.dZ = dA * (self.Z > 0)
        elif self.activation_function == "sigmoid":
            self.dZ = dA * (self.A * (1 - self.A)) 

        self.dW = np.dot(self.A_prev.T, self.dZ) 

        self.db = np.sum(self.dZ, axis=0, keepdims=True) 

        dA_prev = np.dot(self.dZ, self.W.T)

        return dA_prev



NN is the class that connects all the layers together

In [ ]:
class NeuralNetwork:
    def __init__(self):
        self.layers = []

    def add_layer(self, layer):
        self.layers.append(layer)

    def forward(self, X):
        current_input = X 
        for layer in self.layers:
            current_input = layer.forward(current_input)
        return current_input

    def cost_calc(self, A_final, y):
        squared_error = (y - A_final) ** 2
        cost = np.mean(np.sum(squared_error, axis=1))
        return cost

    def backward(self, A_final, y):
        m = y.shape[0]
        dA = (2 / m) * (A_final - y)

        for layer in reversed(self.layers):
            dA = layer.backward(dA)


In [ ]:
n_features = X.shape[1] 
n_outputs = y.shape[1]

layer1 = DenseLayer(n_features, 100, "relu")
layer2 = DenseLayer(100, 100, "relu")
layer3 = DenseLayer(100, n_outputs, "sigmoid")

model = NeuralNetwork()
model.add_layer(layer1)
model.add_layer(layer2)
model.add_layer(layer3)

epoch = 10
learning_rate = 0.05

for i in range(epoch):
    A_final = model.forward(X)
    cost = model.cost_calc(A_final, y)
    print(f"Epoch {i+1}/{epoch} - Cost: {cost}")

    model.backward(A_final,y)

    for layer in model.layers:
        layer.W = layer.W - learning_rate*layer.dW
        layer.b = layer.b - learning_rate*layer.db